# Behavioral questions

## Tell me about a time your AI system didn’t work or gave wrong answers.

In my turbine RAG project, the system initially gave incorrect answers for some troubleshooting queries. I investigated and found the issue was mainly in retrieval quality.

on the retrieval side, we initially used ANN-based search, but later moved to a more precise KNN approach to improve result accuracy. This helped improve the relevance of retrieved chunks.

Top-k retrieved chucks calcueted there average search score and if the retrievev chucks score is less than avarea it will ingored.

On the generation side, I improved the prompt to strictly restrict the model to the provided context, which helped reduce hallucinations. I also implemented confidence filtering—if the retrieval score was below a certain threshold, the system returned a fallback instead of generating a potentially incorrect answer.

As a result, incorrect answers reduced by around 40%, and the system became much more reliable and trustworthy for engineers.

###  Why move from ANN to KNN?

ANN was faster but sometimes returned less accurate neighbors. Since our use case required higher precision for engineering queries, we switched to KNN to improve retrieval accuracy, even with a slight latency trade-off.

## Why did you choose RAG instead of fine-tuning (or vice versa)?

In my use case, I chose RAG over fine-tuning primarily because the data was dynamic and frequently updated, such as turbine manuals and troubleshooting documents. With RAG, we can retrieve the latest information at query time without retraining the model.

Fine-tuning, on the other hand, is better suited for learning patterns or behavior, but it’s static and requires retraining whenever data changes, which is costly and time-consuming.

RAG also provides better transparency since we can show the retrieved sources, which is important for engineering use cases where trust and explainability matter.

So overall, RAG gave us flexibility, lower cost, and more reliable, up-to-date answers compared to fine-tuning.”

###  If They Ask Follow-Up: “When WOULD you use fine-tuning?”

“I would use fine-tuning when the problem requires learning consistent patterns or behavior, like response formatting, tone control, classification, or domain-specific reasoning that retrieval alone cannot handle.”

## How did you reduce latency or improve performance?

To improve latency in my RAG system, I optimized both retrieval and generation. On the retrieval side, I tuned the top-k value and controlled candidate size to balance speed and relevance, especially after moving from ANN to KNN.

On the generation side, I reduced prompt size by passing only the most relevant chunks, which lowered token usage and response time. I also added confidence filtering to avoid unnecessary LLM calls for low-quality queries.

Overall, this reduced unnecessary processing while maintaining answer quality and improving system responsiveness.

### What specific latency improvements did you see?”

“We reduced unnecessary LLM calls for low-confidence queries and optimized retrieval size, which helped improve overall response time while keeping accuracy high.”

### What would you do further to reduce latency?”

“I would add caching for frequent queries, use smaller or faster models where possible, and introduce asynchronous or streaming responses to improve perceived latency.”

## How do you evaluate your AI system?

“I evaluate at two levels—retrieval and generation. For retrieval, I check relevance of top-k results. For generation, I validate correctness and grounding while monitoring hallucinations. I also use manual testing with real queries and compare performance before and after changes.

##  How did you deploy your AI system?
I deployed the AI system using a containerized approach with Docker. The FastAPI backend, which handled the RAG pipeline, was packaged into a container and deployed on Azure App Service.

For CI/CD, I used a pipeline to automatically build and deploy the application whenever changes were pushed, ensuring smooth and consistent releases.

Secrets like API keys and connection strings were managed securely using Azure Key Vault and referenced through managed identity, so nothing was hardcoded.

The system was integrated with Azure AI Search for retrieval and an LLM for generation, and we added logging and monitoring to track performance and errors after deployment.

Overall, this setup ensured scalability, security, and easy maintainability in production.”

### How does the request flow work?”

“User → Frontend → FastAPI API → Retrieval (Azure AI Search) → LLM → Response → User”

### “How did you handle scaling?”

“Azure App Service handled auto-scaling, and since the API is stateless, it scales horizontally easily.”

## Tell me about a data-related issue you faced

I faced an issue where unstructured documents led to poor retrieval quality. I fixed it by improving preprocessing and chunking, making chunks more focused. This significantly improved answer accuracy.”

### “What exactly was wrong with the data?”

“The data had mixed topics in the same sections, so chunks were not semantically clean.”

### “What did you learn?”
“Better data structuring directly improves retrieval, which is critical for RAG performance.”

## Tell me about a trade-off you made (cost vs accuracy, speed vs quality)

In my RAG system, I had to make a trade-off between speed and accuracy during retrieval. Initially, we used ANN-based search, which was fast but sometimes returned less precise results, leading to lower answer quality.

Since our use case involved engineering troubleshooting, accuracy was more critical than speed. So I switched to a KNN-based approach, which improved retrieval precision but increased latency by around 15–20%.

To balance this, I optimized top-k retrieval and added confidence filtering to avoid unnecessary LLM calls for low-quality queries.

This allowed us to prioritize answer quality while keeping latency within an acceptable range

### How did you decide the trade-off?”

“Based on user impact—engineers needed reliable answers more than fast but incorrect ones.”

## Your chatbot is giving wrong answers. What will you do?”

If my chatbot is giving wrong answers, I would debug it systematically across the RAG pipeline.

First, I would check retrieval quality—whether the top-k chunks actually contain relevant information. If not, I’d look at issues like chunking, search strategy, or embeddings.

Second, I’d validate the prompt to ensure the model is strictly grounded in the provided context and not hallucinating.

Third, I’d analyze the retrieved content itself—if the data is noisy or poorly structured, that can lead to incorrect answers.

I’d also check confidence scores and add or tune fallback mechanisms to avoid answering when retrieval is weak.

Finally, I’d use test queries and logs to compare before and after changes to ensure the fixes are actually improving accuracy.

Overall, I focus on identifying whether the issue is in retrieval, data, or generation, and fix it step by step.

## Tell me about a time requirements were unclear. What did you do?

When requirements were unclear, I clarified them by sharing sample outputs and getting feedback from stakeholders. This helped refine expectations and build the system correctly from the start.

### What did you learn?”
“Early feedback is critical, especially in AI systems where expectations can vary widely.”

## System worked fine initially but failed at scale. What happened?

Initially, the system worked well during testing, but when usage increased, we started seeing performance degradation and inconsistent response quality.

After analyzing the issue, I found two main causes. First, the retrieval layer wasn’t optimized for scale—higher query volume increased latency, especially after moving to a more precise KNN-based search. Second, we were sending too many chunks to the LLM, which increased response time and token usage under load.

To fix this, I optimized the top-k retrieval to limit unnecessary data, added stricter filtering to ensure only relevant chunks were passed, and introduced confidence-based early exits to reduce unnecessary LLM calls. I also ensured the API was stateless so it could scale horizontally.

After these changes, the system handled higher load more efficiently with improved response consistency.

## Tell me about a time you had to coordinate across different technical areas or teammates with different expertise. How did you ensure everyone stayed aligned?

I aligned cross-functional teams by defining clear API contracts, sharing sample outputs including edge cases, and maintaining regular sync-ups. This ensured smooth integration and avoided miscommunication.


## Describe a time you had to bridge the gap between a technical solution you built and a non-technical stakeholder. How did you translate the system's behavior into something meaningful for them?

I explained the AI system as a ‘smart search + summarization tool’ instead of using technical terms, and used real examples to show its behavior. This helped non-technical stakeholders understand and trust the system.


### What was the biggest challenge?
“Explaining why the system sometimes doesn’t answer—people expect AI to always respond.


## Describe a real debugging scenario where a silent failure caused unexpected behavior — like Azure AI Search returning fewer than K results — and how you built a safeguard.

I encountered a silent failure where Azure AI Search returned fewer than top-k results. I added safeguards by checking result count and average similarity score, and triggered fallback responses when confidence was low, preventing incorrect answers.

## Tell me about a time you encountered a subtle bug like a cosine similarity false positive. What was your debugging process?

“In my RAG system, I encountered a subtle issue where cosine similarity was returning false positives—chunks that were semantically similar at a high level but not actually relevant to the user’s query. This caused the model to generate partially incorrect answers even though retrieval scores looked high.

To debug this, I first logged the top-k retrieved chunks along with their similarity scores and manually inspected them. I noticed that some chunks shared common keywords or general context but didn’t answer the specific query.

I then validated this by testing with multiple queries and confirmed that high similarity didn’t always mean high relevance.

To fix it, I introduced stricter filtering using an average similarity threshold and limited the number of chunks passed to the model. I also improved the prompt to focus on precise context.

This reduced false positives and improved answer accuracy significantly.

This experience taught me that similarity scores alone aren’t enough—you need additional validation to ensure true relevance

### What would you do further?
Add re-ranking (cross-encoder) or metadata filtering to improve precision.

## Tell me about a time you had to diagnose a latency issue in a multi-service pipeline. How did you isolate the bottleneck?

In my RAG-based system, we observed increased response latency in production as usage grew. Since the pipeline involved multiple services—API layer, retrieval, and LLM—I needed to isolate where the delay was coming from.

I started by adding detailed logging and timing at each stage of the pipeline—API processing, Azure AI Search retrieval, and LLM response time. This helped me break down the end-to-end latency.

From this, I identified that retrieval latency had increased significantly, especially after switching from ANN to KNN, and we were also passing too many chunks to the LLM, which increased generation time.

To fix this, I optimized top-k retrieval to limit unnecessary data, reduced the number of chunks sent to the model, and added early fallback for low-confidence queries to avoid unnecessary LLM calls.

After these changes, overall latency improved and became more consistent across requests.

### What tools did you use?”
“Primarily logging and custom timing metrics; in production, this can be extended with monitoring tools.


## Tell me about a time you took full ownership of a complex system. What decisions did you make independently, and where did you seek input?

I owned the end-to-end RAG system, making key decisions on architecture, retrieval, and prompt design. At the same time, I collaborated with domain experts, DevOps, and frontend teams to ensure accuracy and smooth deployment.


## Walk me through a time you had to make an important design decision with incomplete information. How did you validate it and what would you do differently now?

In my RAG-based chatbot project, one key design decision I had to make with incomplete information was choosing between a pure keyword-based retrieval system and a more advanced semantic retrieval approach. At the early stage, we didn’t have enough real user queries or evaluation data to confidently decide which would perform better.

Given the uncertainty, I started with a simpler keyword-based approach to move quickly, but I designed the system in a modular way so we could easily switch or extend the retrieval strategy later.

To validate the decision, I created a small set of representative test queries and compared retrieval relevance manually. Based on the results, I observed that keyword search alone was not sufficient for semantic queries, so I iteratively introduced vector search and moved to a hybrid approach.

If I were to do it again, I would invest earlier in building a structured evaluation dataset and metrics like precision@k, so decisions could be more data-driven from the beginning.

This experience taught me the importance of designing for flexibility when working with uncertainty.

## Tell me about a time you identified a technical limitation in your own work mid-project. How did you balance fixing it versus shipping on time?

During my RAG chatbot project, midway through development I identified a limitation in the retrieval system. Initially, we were relying on ANN-based search, which was fast but sometimes returned less precise results, affecting answer quality.

As I tested the system with real queries, I realized that improving retrieval accuracy would require switching to a more precise KNN-based approach and refining chunking, which could impact latency and required additional effort.

At that point, I had to balance fixing it versus shipping on time. I evaluated the impact and decided to prioritize a practical approach: I improved the prompt to better ground responses and introduced confidence filtering to reduce incorrect answers, while planning retrieval improvements as a follow-up enhancement.

This allowed us to meet the delivery timeline while still improving reliability. Later, we incrementally improved retrieval by tuning top-k and similarity thresholds.

So overall, I focused on delivering a stable version first, while scheduling deeper optimizations as iterative improvements.

### Why didn’t you fix retrieval immediately?
Because retrieval changes were more time-consuming and could impact latency, so I chose a quicker mitigation first to meet deadlines.


## Tell me about a time your AI system produced a confident but wrong answer in production. How did you respond and what guardrails did you put in place?

In my RAG-based chatbot, we observed an issue in production where the system was producing confident but incorrect answers for some troubleshooting queries. The responses sounded plausible, but when verified against the source documents, they were not accurate.

I investigated the issue by reviewing logs and inspecting the retrieved chunks for those queries. I found that in some cases, the retrieval was returning weak or partially relevant context, but the LLM was still generating a fluent answer based on that incomplete information.

To address this, I introduced multiple guardrails. First, I strengthened the prompt to strictly instruct the model to answer only from the provided context and to avoid guessing. Second, I implemented confidence filtering based on retrieval scores and average similarity across top-k results—if the confidence was below a threshold, the system returned a fallback response instead of generating an answer.

Additionally, I limited the amount of retrieved context passed to the model to reduce noise and improve grounding.

These changes significantly reduced hallucinations and ensured that the system either provided a reliable answer or gracefully indicated when it didn’t have enough information.


### How did you detect it?
Through logs, user feedback, and manual validation of retrieved chunks versus generated answers.

### What guardrails are most important?
Prompt grounding, retrieval quality checks, and confidence-based fallback mechanisms.

## Describe a time a system you built broke under real-world load or usage patterns you hadn't predicted. What did you learn?

In my RAG-based chatbot system, everything worked well during development and testing with a limited number of queries. However, when the system was exposed to real-world usage patterns, we started seeing issues with both latency and answer quality under higher load and more diverse queries.

One key issue was that some queries were much more ambiguous or complex than our test cases, which led to poorer retrieval results. In addition, the system was retrieving and processing more chunks than necessary, which increased response time under load.

To address this, I optimized the retrieval pipeline by tuning the top-k value and limiting the number of chunks passed to the LLM. I also introduced confidence filtering so that low-quality queries would return a fallback instead of consuming full pipeline resources.

This experience taught me that real-world usage is often more unpredictable than test scenarios, and it’s important to design systems with variability, noisy inputs, and scalability in mind from the beginning.

### What would you do differently?
I would include more realistic test queries, load testing, and monitoring from the beginning.

## Tell me about a time a technical choice you strongly advocated for turned out to be the wrong one. How did you recognize it and what did you do?

In my RAG project, I initially advocated for using ANN-based retrieval because it offered lower latency and was expected to scale better for our use case. At first, it seemed like the right choice since performance was good during initial testing.

However, as we started testing with real engineering queries, we noticed that the answer quality was inconsistent—some retrieved chunks were not truly relevant, even though they had acceptable similarity scores. This led to incorrect or partially correct answers.

I recognized this through user feedback and by manually inspecting retrieval outputs, where I saw that ANN was sometimes returning approximate neighbors that weren’t precise enough for our domain.

After analyzing the trade-offs, I proposed moving to a more precise KNN-based approach, even though it came with a slight latency increase. We also complemented it with better scoring thresholds and filtering.

Once we made the switch, retrieval relevance improved significantly, and the overall system accuracy increased, even though we accepted a small performance trade-off. This taught me that in some cases, accuracy and relevance should be prioritized over raw performance.

### What would you do differently now?
I would evaluate retrieval strategies earlier using real test queries before finalizing the approach.


## intrudcation 

Hi, first of all, thank you for the opportunity.
My name is Sanjay, and I’m from Bengaluru. I completed my Bachelor’s degree in Mechanical Engineering.

I currently work as an AI Engineer at KPMG, and I have over 3 years of industry experience, including 2+ years working on Generative AI and LLM-based systems, mainly focused on RAG applications.

Additionally, I have domain experience in healthcare and renewable energy.

## workflow

I work on designing and building RAG-based AI systems.

My responsibilities include developing APIs using FastAPI, integrating Azure OpenAI and Azure AI Search, and building end-to-end pipelines to process user queries.

I also focus on improving system performance by optimizing retrieval quality, reducing latency, and handling rate limits.

Additionally, I monitor system performance, debug issues, and ensure the system is scalable and reliable.

## Why are you switching the company?

I’ve had a great experience at KPMG working on AI systems. Now, I’m looking for more challenging, large-scale work where I can contribute to end-to-end system design and have a bigger impact.


## Why not stay and grow in your current company?

“I’ve definitely considered growth within KPMG, and I’ve had a good learning experience working on AI systems like RAG pipelines and deployment.

While internal growth is possible, at this point I’m looking for opportunities where I can work on more complex and larger-scale systems, with deeper involvement in architecture decisions and end-to-end ownership. I felt that this role aligns more directly with those goals and the kind of challenges I want to take on next

## Did you try for internal project switching?


“Yes, I’ve explored internal opportunities within KPMG. Internal mobility is possible, but it depends on project availability, approvals, and timelines, and sometimes the opportunities don’t align exactly with the kind of work or scale I’m looking for.

That’s why I decided to explore external opportunities that better match my interest in working on more advanced AI systems and broader responsibilities.

## What limitations are you facing in your current role?

“I wouldn’t call them limitations, but rather the natural scope of the role at my current experience level.

I’ve gained solid hands-on experience in building and deploying RAG-based systems, but most of the work has been within a certain scope. I’m now looking to expand into areas like deeper system design, scaling architectures, performance optimization, and handling more complex production challenges.

So it’s more about seeking broader exposure rather than any gap or issue in my current role.”

## What exactly are you looking for in your next role?


“In my next role, I’m looking to work on more challenging AI systems where I can contribute beyond implementation — especially in areas like system design, optimization, and improving end-to-end performance.

I’m also looking for opportunities to work on production-grade systems at scale, collaborate with cross-functional teams, and take more ownership of features or components. Overall, I want a role that helps me grow from a hands-on engineer into someone who can also think about architecture and system-level decisions

## Why this company specifically?

I was particularly interested in this opportunity because it aligns well with my experience and the direction I want to grow in. The role involves working on AI systems at a larger scale, which matches my background in RAG pipelines and retrieval systems, and also gives me the chance to expand into more complex system design and optimization work.

Additionally, I see this as an environment where I can continue learning, take on more responsibility, and contribute meaningfully while growing as an AI engineer.

## What are your long-term career goals?

I see myself growing into a more experienced AI engineer who can design and build scalable AI systems end-to-end. Over the next few years, I want to strengthen my skills in areas like system design, retrieval optimization, performance tuning, and working with large-scale AI architectures.

I also want to take on more ownership of components or systems, contribute to architectural decisions, and collaborate closely with cross-functional teams.

In the long term, I aim to evolve into a role where I can not only implement solutions but also help design robust AI systems and guide technical decisions in complex projects.”